# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [ ]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [ ]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [ ]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [ ]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [ ]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


TrainOutput(global_step=690, training_loss=0.33479292703711466, metrics={'train_runtime': 80.2819, 'train_samples_per_second': 137.067, 'train_steps_per_second': 8.595, 'total_flos': 428577075854640.0, 'train_loss': 0.33479292703711466, 'epoch': 3.0})

In [ ]:
model_alt = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

training_args_alt = TrainingArguments(
    output_dir="bert-mrpc-8-epochs",
    num_train_epochs=8,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer_alt = Trainer(
    model=model_alt,
    args=training_args_alt,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Параметрів, що тренуються: {count_params(model_alt, trainable_only=True):,}")
trainer_alt.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.550131,0.401439,0.818627,0.872852
2,0.316217,0.423148,0.821078,0.880524
3,0.142796,0.515506,0.855392,0.895204
4,0.060471,0.677285,0.860294,0.903226
5,0.026307,0.772317,0.865196,0.904014
6,0.016525,0.863644,0.850490,0.896435
7,0.010557,0.875014,0.850490,0.895726
8,0.008447,0.831392,0.862745,0.902778


TrainOutput(global_step=1840, training_loss=0.14143122499403746, metrics={'train_runtime': 160.0348, 'train_samples_per_second': 183.36, 'train_steps_per_second': 11.498, 'total_flos': 1146185757132480.0, 'train_loss': 0.14143122499403746, 'epoch': 8.0})

In [ ]:
results_alt = trainer_alt.evaluate()
results = trainer.evaluate()

comparison = pd.DataFrame([
    {
        "Модель": "BERT (3 епохи)",
        "Loss": results["eval_loss"],
        "Accuracy": results["eval_accuracy"],
        "F1": results["eval_f1"],
    },
    {
        "Модель": "BERT (8 епох)",
        "Loss": results_alt["eval_loss"],
        "Accuracy": results_alt["eval_accuracy"],
        "F1": results_alt["eval_f1"],
    }
])

display(comparison)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.008447,0.831392,8,0.862745,0.902778


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


,Модель,Loss,Accuracy,F1
0,BERT (3 епохи),0.416541,0.860294,0.901554
1,BERT (8 епох),0.831392,0.862745,0.902778


## 4. Перевірка на власних прикладах

In [ ]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]

    return f"парафраз: {probs[1]:.1%}"

pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    ("The meeting was postponed until Friday.", "The meeting has been delayed to Friday."),
    ("A dog is running through the grass.", "A canine is sprinting across a lawn."),
    ("The restaurant is closed on Mondays.", "The museum offers free admission on Sundays."),
    ("The student submitted the assignment before the deadline.", "The assignment was turned in by the student ahead of time."),
]

for i, (s1, s2) in enumerate(pairs, start=1):
    print(f"\nПеревірка {i}:")
    print(f"Речення 1: {s1}")
    print(f"Речення 2: {s2}")
    print(f"Модель (3 епохи): {predict(model, s1, s2)}")
    print(f"Модель (8 епох): {predict(model_alt, s1, s2)}")
    print()



Перевірка 1:
Речення 1: The company reported a profit of $5 million.
Речення 2: The firm earned five million dollars in profit.
Модель (3 епохи): парафраз: 86.0%
Модель (8 епох): парафраз: 99.9%


Перевірка 2:
Речення 1: The company reported a profit of $5 million.
Речення 2: The weather was sunny all week.
Модель (3 епохи): парафраз: 2.6%
Модель (8 епох): парафраз: 0.1%


Перевірка 3:
Речення 1: She bought a new car yesterday.
Речення 2: Yesterday she purchased a new vehicle.
Модель (3 епохи): парафраз: 99.0%
Модель (8 епох): парафраз: 100.0%


Перевірка 4:
Речення 1: The meeting was postponed until Friday.
Речення 2: The meeting has been delayed to Friday.
Модель (3 епохи): парафраз: 98.8%
Модель (8 епох): парафраз: 100.0%


Перевірка 5:
Речення 1: A dog is running through the grass.
Речення 2: A canine is sprinting across a lawn.
Модель (3 епохи): парафраз: 98.7%
Модель (8 епох): парафраз: 100.0%


Перевірка 6:
Речення 1: The restaurant is closed on Mondays.
Речення 2: The museum o

# ВІДПОВІДІ

1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?
2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?
3. Яке пряме застосування базової моделі bert-base-uncased?
4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?
5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?
6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?
7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?
8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?
9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.
10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

1. Клітинки виконувалися без суттєвих затримок.\
Завантаження датасету MRPC та моделі BERT пройшло успішно:
+ Навчання моделі на 3 епохи тривало приблизно 1 хвилину 20 секунд.
+ Навчання моделі на 8 епох тривало приблизно 2 хвилини 40 секунд.\
\
Під час запуску виникали незначні попередження: відсутність токена авторизації на Hugging Face, а також повідомлення MISSING та UNEXPECTED щодо параметрів моделі. Ці повідомлення не перешкоджали виконанню ноутбуку.\
Загалом критичних проблем із запуском не виникло, всі клітинки виконалися успішно.
2. Це спеціальний токен, який додається на початку тексту. Він допомагає моделі BERT зібрати загальну інформацію про все речення. Після обробки тексту отримане представлення [CLS] передається до класифікаційного шару, який приймає рішення.
3. Базова модель bert-base-uncased призначена для обробки та розуміння тексту англійською мовою. Вона попередньо навчена розпізнавати контекст слів у реченнях.\
Її можна використовувати як основу для подальшого навчання під різні завдання, пов'язані з розумінням та роботою з текстом.
4. З проявами сексизму в LLM-моделях потрібно боротися, оскільки штучний інтелект навчається на текстах, створених людьми та переймає відповідні стереотипи та упередження.\
Для зменшення таких проявів потрібно використовувати різноманітні та збалансовані набори даних для навчання, перевіряти моделі на наявність упереджень та фільтрувати дані. Також можна використовувати існуючі мовні моделі з відповідними промптами для швидшої курації даних.
5. Це значення token_type_ids, які показують, до якого з двох речень належить кожен токен:\
0 позначає токени першого речення, 1 - другого.\
Це потрібно моделі для того, щоб розрізняти два речення та правильно аналізувати їхній взаємозв'язок.
6. Число 768 - це розмір векторного представлення тексту, який формує модель BERT. Воно утворюється, тому що BERT використовує 12 голів, кожна з яких працює з розміром вектора 64: `12 * 64 = 768`.\
Число 2 - це кількість класів, між якими модель обирає: 0 - речення нееквівалентні, 1 - речення еквівалентні. Linear(in_features=768, out_features=2) означає, що модель бере вхідний вектор із 768 значень і на його основі визначає, до якого з двох класів належить пара речень.
7. Це добре, оскільки основна частина параметрів знаходиться в backbone, який уже навчений розуміти мову та контекст слів. Голова має значно менше параметрів і відповідає лише за класифікацію речень на два класи: еквівалентні та нееквівалентні. Завдяки цьому не потрібно навчати модель розуміти мову з нуля, а достатньо адаптувати вже готову модель під певну задачу.
8. `Accuracy` - це метрика, що визначає відсоток правильно класифікованих пар речень від їх загальної кількості. Вона враховує як правильно визначені еквівалентні пари, так і правильно визначені нееквівалентні пари.\
`F1-score` - це метрика, яка враховує два показники:\
`Precision`: частка правильних серед усіх пар, які модель визначила як еквівалентні.\
`Recall`: частка знайдених моделлю еквівалентних пар серед усіх фактично еквівалентних пар.\
`F1-score` є їхнім гармонійним середнім.
9. Зі збільшенням кількості епох модель краще запам'ятовує навчальні дані, тому Training Loss постійно зменшується.\
Validation Loss після третьої епохи починає зростати, що свідчить про перенавчання, так як модель починає запам'ятовувати конкретні приклади замість того, щоб вивчати закономірності.\
Accuracy та F1 майже не покращилися.

3 епохи

| Epoch | Training Loss | Validation Loss | Accuracy | F1       |
|------:|--------------:|----------------:|---------:|---------:|
| 1     | 0.487028      | 0.381455        | 0.838235 | 0.886986 |
| 2     | 0.328676      | 0.357605        | 0.848039 | 0.893471 |
| 3     | 0.167169      | 0.416541        | 0.860294 | 0.901554 |

8 епох

| Epoch | Training Loss | Validation Loss | Accuracy | F1       |
|------:|--------------:|----------------:|---------:|---------:|
| 1     | 0.550131      | 0.401439        | 0.818627 | 0.872852 |
| 2     | 0.316217      | 0.423148        | 0.821078 | 0.880524 |
| 3     | 0.142796      | 0.515506        | 0.855392 | 0.895204 |
| 4     | 0.060471      | 0.677285        | 0.860294 | 0.903226 |
| 5     | 0.026307      | 0.772317        | 0.865196 | 0.904014 |
| 6     | 0.016525      | 0.863644        | 0.850490 | 0.896435 |
| 7     | 0.010557      | 0.875014        | 0.850490 | 0.895726 |
| 8     | 0.008447      | 0.831392        | 0.862745 | 0.902778 |

| Метрика          | 3 епохи  | 8 епох   |
|------------------|---------:|---------:|
| Training Loss    | 0.167169 | 0.008447 |
| Validation Loss  | 0.416541 | 0.831392 |
| Accuracy         | 86.03%   | 86.27%   |
| F1               | 90.16%   | 90.28%   |

10. *Додано в клітинці вище.*